# California Housing Prices Analysis

## 1. Understanding Data and Problem Statement

This notebook analyzes the California Housing Prices dataset to build a regression model for predicting house prices. We'll go through the complete data modeling process including data preparation, model training, and addressing overfitting/underfitting issues.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn')
sns.set_palette('husl')

### Load and Examine the Data

In [ ]:
# Load the dataset
df = pd.read_csv('housing.csv')

# Display basic information about the dataset
print("Dataset Info:")
print("=============\n")
print(df.info())
print("\nSample Data:")
print("============\n")
print(df.head())
print("\nBasic Statistics:")
print("================\n")
print(df.describe())

## 2. Data Preparation and Feature Selection

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Handle missing values if any
df = df.fillna(df.mean())

# Create correlation matrix heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

# Distribution of target variable (median_house_value)
plt.figure(figsize=(10, 6))
sns.histplot(df['median_house_value'], bins=50)
plt.title('Distribution of Median House Values')
plt.show()

### Feature Engineering and Preprocessing

In [ ]:
# Create new features
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

# Prepare features and target
X = df.drop(['median_house_value'], axis=1)
y = df['median_house_value']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

## 3. Model Training and Testing

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'MSE': mse,
        'R2': r2
    }

# Display results
print("Model Performance:")
print("=================\n")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"MSE: {metrics['MSE']:.2f}")
    print(f"R2: {metrics['R2']:.2f}\n")

## 4. Addressing Overfitting and Underfitting

In [ ]:
# Perform k-fold cross-validation
def evaluate_model(model, X, y, cv=5):
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='r2')
    return cv_scores.mean(), cv_scores.std()

print("Cross-Validation Results:")
print("=======================\n")

for name, model in models.items():
    mean_score, std_score = evaluate_model(model, X_scaled, y)
    print(f"{name}:")
    print(f"Mean R2: {mean_score:.2f} (+/- {std_score*2:.2f})\n")

# Try different regularization strengths for Ridge regression
alphas = [0.1, 1.0, 10.0, 100.0]
ridge_scores = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    mean_score, _ = evaluate_model(ridge, X_scaled, y)
    ridge_scores.append(mean_score)

plt.figure(figsize=(10, 6))
plt.plot(alphas, ridge_scores, marker='o')
plt.xscale('log')
plt.xlabel('Alpha (regularization strength)')
plt.ylabel('Mean R2 score')
plt.title('Ridge Regression Performance vs Regularization Strength')
plt.grid(True)
plt.show()

## 5. Reflection and Insights

### Key Findings:
1. The models' performance metrics indicate how well they can predict housing prices
2. Cross-validation results show the models' generalization capabilities
3. Regularization helps prevent overfitting by controlling model complexity

### Real-world Applications:
- Real estate price estimation
- Market analysis for property developers
- Investment decision support

### Limitations:
1. The model assumes a static market condition
2. Local market variations might not be captured
3. External factors (economic conditions, policy changes) are not considered

### Future Improvements:
1. Include more features (e.g., crime rates, school ratings)
2. Try more advanced models (e.g., XGBoost, Random Forests)
3. Incorporate time-series analysis for price trends